In [ ]:
# @title 1.1 🔍 Check GPU
import torch

print("🔍 GPU Check:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name} ({mem_gb:.1f} GB)")
else:
    print("❌ GPU not found! Enable GPU in Kaggle Settings.")

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
import os
import sys

REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"
WORK_DIR = "/kaggle/working/TriMedAgent"

if not os.path.exists(WORK_DIR):
    print("📥 Cloning TriMedAgent...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print("✅ Repository exists.")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Install dependencies (standard transformers + peft, NOT unsloth)
print("\n📦 Installing training dependencies...")
!pip install -q transformers>=4.36.0
!pip install -q peft>=0.7.0
!pip install -q accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q datasets wandb huggingface_hub
!pip install -q trl>=0.7.0

print("\n✅ Installation Complete!")
print("   Using: transformers + peft (standard training)")

In [ ]:
# @title 1.3 🔑 Setup HuggingFace Token
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Get token from Kaggle secrets
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace!")
except Exception as e:
    print("⚠️ Could not get HF_TOKEN from secrets.")
    print("   Add HF_TOKEN to Kaggle Secrets, or login manually:")
    HF_TOKEN = input("Enter HuggingFace Token: ")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("✅ Logged in!")

---
## 2️⃣ 📊 Download & Preprocess Training Data

**Data Sources cho SFT:**
- **VQA-RAD**: Visual Question Answering in Radiology
- **SLAKE**: Structured Medical Knowledge VQA
- **VinDr-CXR**: Vietnamese X-ray with Detection Boxes

**Output Format (ReAct-style):**
```json
{
    "image": "path/to/img.jpg",
    "conversations": [
        {"role": "user", "content": "<image>\nQuestion"},
        {"role": "assistant", "content": "{\"thought\": \"...\", \"action\": \"...\", \"action_input\": {...}}"}
    ]
}
```

In [ ]:
# @title 2.1 📥 Download VQA-RAD Dataset
import os
import json
import requests
import zipfile
from pathlib import Path

DATA_DIR = Path("data")
SFT_DIR = DATA_DIR / "sft_dataset"
RAW_DIR = DATA_DIR / "raw"
SFT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("📥 DOWNLOADING VQA-RAD DATASET")
print("="*60)

# VQA-RAD từ OSF
VQA_RAD_URL = "https://osf.io/89kps/download"

vqa_rad_path = RAW_DIR / "VQA_RAD"
if not vqa_rad_path.exists():
    print("📥 Downloading VQA-RAD from OSF...")
    print("   URL: https://osf.io/89kps/")
    
    # Download zip
    zip_path = RAW_DIR / "vqa_rad.zip"
    
    try:
        # Thử download trực tiếp
        !wget -q -O {zip_path} "https://osf.io/89kps/download" || echo "wget failed, trying curl..."
        !curl -L -o {zip_path} "https://osf.io/89kps/download" 2>/dev/null || echo "Direct download may require manual download"
        
        if zip_path.exists() and zip_path.stat().st_size > 1000:
            print("📦 Extracting...")
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(RAW_DIR)
            print("✅ VQA-RAD downloaded!")
        else:
            print("⚠️ Auto-download failed. Manual download required:")
            print("   1. Go to: https://osf.io/89kps/")
            print("   2. Download VQA_RAD.zip")
            print("   3. Extract to: data/raw/VQA_RAD/")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("⚠️ Please download manually from https://osf.io/89kps/")
else:
    print("✅ VQA-RAD already exists!")
    
# List contents
if vqa_rad_path.exists():
    print(f"\n📁 Contents of {vqa_rad_path}:")
    for item in list(vqa_rad_path.iterdir())[:10]:
        print(f"   - {item.name}")

In [ ]:
# @title 2.2 📥 Download SLAKE Dataset (Kaggle)
from pathlib import Path

print("="*60)
print("📥 DOWNLOADING SLAKE DATASET")
print("="*60)

SLAKE_DIR = RAW_DIR / "SLAKE"

if not SLAKE_DIR.exists():
    print("📥 Downloading SLAKE from Kaggle...")
    print("   Dataset: awsaf49/slake-vqa-dataset")
    
    try:
        # Kaggle API
        !pip install -q kaggle
        !mkdir -p ~/.kaggle
        
        # Check if kaggle.json exists
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_json.exists():
            print("\n⚠️ Kaggle credentials required!")
            print("   1. Go to: https://www.kaggle.com/settings")
            print("   2. Create New API Token → downloads kaggle.json")
            print("   3. Upload kaggle.json to Kaggle notebook")
            print("   4. Or copy content below:")
            
            # Try Kaggle secrets
            try:
                from kaggle_secrets import UserSecretsClient
                secrets = UserSecretsClient()
                kaggle_user = secrets.get_secret("KAGGLE_USERNAME")
                kaggle_key = secrets.get_secret("KAGGLE_KEY")
                
                with open(kaggle_json, 'w') as f:
                    json.dump({"username": kaggle_user, "key": kaggle_key}, f)
                !chmod 600 ~/.kaggle/kaggle.json
                print("✅ Kaggle credentials loaded from secrets!")
            except:
                print("   Add KAGGLE_USERNAME and KAGGLE_KEY to Kaggle Secrets")
        
        # Download SLAKE
        !kaggle datasets download -d awsaf49/slake-vqa-dataset -p {RAW_DIR} --unzip
        
        # Rename if needed
        slake_downloaded = RAW_DIR / "slake-vqa-dataset"
        if slake_downloaded.exists():
            slake_downloaded.rename(SLAKE_DIR)
            
        print("✅ SLAKE downloaded!")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\n📋 Manual download:")
        print("   1. Go to: https://www.kaggle.com/datasets/awsaf49/slake-vqa-dataset")
        print("   2. Download and extract to: data/raw/SLAKE/")
else:
    print("✅ SLAKE already exists!")

# Show contents
if SLAKE_DIR.exists():
    print(f"\n📁 Contents of {SLAKE_DIR}:")
    for item in list(SLAKE_DIR.iterdir())[:10]:
        print(f"   - {item.name}")

In [ ]:
# @title 2.3 🔄 Preprocess VQA-RAD → ReAct Format
import json
from pathlib import Path

print("="*60)
print("🔄 PREPROCESSING VQA-RAD → ReAct Format")
print("="*60)

def get_action_for_question(question: str) -> dict:
    """Determine which tool to use based on question type"""
    q_lower = question.lower()
    
    # Detection keywords
    detect_keywords = ['where', 'locate', 'find', 'show', 'point', 'identify', 'detect',
                       'ở đâu', 'tìm', 'chỉ', 'xác định', 'vị trí']
    
    # Description keywords  
    describe_keywords = ['what', 'describe', 'how', 'appear', 'look', 'observe',
                        'gì', 'mô tả', 'như thế nào', 'nhận xét', 'đánh giá']
    
    # Yes/No keywords
    yesno_keywords = ['is there', 'are there', 'does', 'do', 'can you see', 'có', 'không']
    
    # Anatomy keywords for detection
    anatomy = ['lung', 'heart', 'chest', 'bone', 'liver', 'kidney', 'brain',
               'phổi', 'tim', 'ngực', 'xương', 'gan', 'thận', 'não']
    
    # Abnormality keywords
    abnormal = ['nodule', 'mass', 'tumor', 'lesion', 'abnormal', 'opacity', 'effusion',
                'nốt', 'khối', 'u', 'tổn thương', 'bất thường', 'mờ', 'tràn']
    
    has_anatomy = any(kw in q_lower for kw in anatomy)
    has_abnormal = any(kw in q_lower for kw in abnormal)
    has_detect = any(kw in q_lower for kw in detect_keywords)
    has_describe = any(kw in q_lower for kw in describe_keywords)
    has_yesno = any(kw in q_lower for kw in yesno_keywords)
    
    # Decide action
    if has_detect or (has_abnormal and has_anatomy):
        # Use GroundingDINO for detection
        prompt = extract_target(question)
        return {
            "thought": f"Cần detect/locate '{prompt}' trong ảnh",
            "action": "GroundingDINO", 
            "action_input": {"prompt": prompt}
        }
    elif has_describe:
        # Use LLaVA for description
        return {
            "thought": "Cần phân tích và mô tả ảnh y tế",
            "action": "LLaVA",
            "action_input": {"question": question}
        }
    elif has_yesno:
        # Use LLaVA for yes/no
        return {
            "thought": "Đây là câu hỏi yes/no, cần quan sát ảnh",
            "action": "LLaVA", 
            "action_input": {"question": question}
        }
    else:
        # Default: LLaVA for general analysis
        return {
            "thought": "Phân tích ảnh y tế để trả lời câu hỏi",
            "action": "LLaVA",
            "action_input": {"question": question}
        }

def extract_target(question: str) -> str:
    """Extract detection target from question"""
    targets = {
        'nodule': 'lung nodule',
        'mass': 'lung mass',
        'tumor': 'tumor mass',
        'heart': 'heart cardiac',
        'lung': 'lung',
        'effusion': 'pleural effusion',
        'pneumonia': 'pneumonia consolidation',
        'fracture': 'bone fracture',
        'cardiomegaly': 'enlarged heart',
    }
    q_lower = question.lower()
    for key, val in targets.items():
        if key in q_lower:
            return val
    return 'abnormality'

def process_vqa_rad(vqa_rad_path: Path) -> list:
    """Process VQA-RAD JSON files to ReAct format"""
    examples = []
    
    # Try different possible file locations
    possible_files = [
        vqa_rad_path / "VQA_RAD Dataset Public.json",
        vqa_rad_path / "trainset.json",
        vqa_rad_path / "testset.json",
    ]
    
    for json_file in possible_files:
        if json_file.exists():
            print(f"📖 Reading: {json_file.name}")
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if isinstance(data, list):
                items = data
            elif isinstance(data, dict):
                items = list(data.values()) if not isinstance(list(data.values())[0], dict) else data.get('data', [])
            else:
                continue
                
            for item in items:
                try:
                    question = item.get('question', item.get('Question', ''))
                    answer = item.get('answer', item.get('Answer', ''))
                    image = item.get('image_name', item.get('Image', item.get('image', '')))
                    
                    if not question:
                        continue
                    
                    # Generate ReAct plan
                    plan = get_action_for_question(question)
                    
                    example = {
                        "image": str(image) if image else None,
                        "conversations": [
                            {"role": "user", "content": f"<image>\n{question}"},
                            {"role": "assistant", "content": json.dumps(plan, ensure_ascii=False)}
                        ],
                        "ground_truth_answer": answer
                    }
                    examples.append(example)
                except Exception as e:
                    continue
                    
            print(f"   → Processed {len(examples)} examples")
    
    return examples

# Process VQA-RAD
VQA_RAD_PATH = RAW_DIR / "VQA_RAD"
vqa_rad_examples = []

if VQA_RAD_PATH.exists():
    vqa_rad_examples = process_vqa_rad(VQA_RAD_PATH)
    print(f"\n✅ VQA-RAD: {len(vqa_rad_examples)} examples")
else:
    print("⚠️ VQA-RAD not found. Using sample data instead.")
    
# Show sample
if vqa_rad_examples:
    print("\n📋 Sample processed example:")
    sample = vqa_rad_examples[0]
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:500])

In [ ]:
# @title 2.4 🔄 Preprocess SLAKE → ReAct Format
import json
from pathlib import Path

print("="*60)
print("🔄 PREPROCESSING SLAKE → ReAct Format")
print("="*60)

def process_slake(slake_path: Path) -> list:
    """Process SLAKE dataset to ReAct format"""
    examples = []
    
    # SLAKE has train.json, validate.json, test.json
    for split in ['train', 'validate', 'test']:
        json_file = slake_path / f"{split}.json"
        if not json_file.exists():
            # Try alternative path
            json_file = slake_path / "Slake1.0" / f"{split}.json"
        
        if json_file.exists():
            print(f"📖 Reading: {json_file.name}")
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            for item in data:
                try:
                    question = item.get('question', '')
                    answer = item.get('answer', '')
                    image = item.get('img_name', item.get('image', ''))
                    q_type = item.get('q_lang', 'en')  # en or zh
                    
                    if not question or q_type == 'zh':  # Skip Chinese for now
                        continue
                    
                    # Generate ReAct plan
                    plan = get_action_for_question(question)
                    
                    example = {
                        "image": str(image) if image else None,
                        "conversations": [
                            {"role": "user", "content": f"<image>\n{question}"},
                            {"role": "assistant", "content": json.dumps(plan, ensure_ascii=False)}
                        ],
                        "ground_truth_answer": answer,
                        "content_type": item.get('content_type', 'unknown'),
                        "answer_type": item.get('answer_type', 'unknown')
                    }
                    examples.append(example)
                except Exception as e:
                    continue
            
            print(f"   → Total so far: {len(examples)} examples")
    
    return examples

# Process SLAKE
SLAKE_PATH = RAW_DIR / "SLAKE"
slake_examples = []

if SLAKE_PATH.exists():
    slake_examples = process_slake(SLAKE_PATH)
    print(f"\n✅ SLAKE: {len(slake_examples)} examples")
else:
    print("⚠️ SLAKE not found. Continuing without it.")

# Show statistics
if slake_examples:
    from collections import Counter
    content_types = Counter(ex.get('content_type', 'unknown') for ex in slake_examples)
    print("\n📊 SLAKE Content Types:")
    for ct, count in content_types.most_common(10):
        print(f"   - {ct}: {count}")

In [ ]:
# @title 2.5 📦 Combine & Save SFT Dataset
import json
import random
from pathlib import Path

print("="*60)
print("📦 COMBINING ALL DATA → SFT DATASET")
print("="*60)

# Combine all examples
all_examples = []

# Add VQA-RAD
if vqa_rad_examples:
    all_examples.extend(vqa_rad_examples)
    print(f"✅ Added VQA-RAD: {len(vqa_rad_examples)} examples")

# Add SLAKE
if slake_examples:
    all_examples.extend(slake_examples)
    print(f"✅ Added SLAKE: {len(slake_examples)} examples")

# If no real data, use sample data
if len(all_examples) < 100:
    print("\n⚠️ Insufficient real data. Adding sample data for demo...")
    
    SAMPLE_DATA = [
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nTìm các nốt mờ bất thường trong phổi."},
                {"role": "assistant", "content": '{"thought": "Cần detect nodule trong phổi", "action": "GroundingDINO", "action_input": {"prompt": "lung nodule"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nPhân tích ảnh X-quang ngực này."},
                {"role": "assistant", "content": '{"thought": "Phân tích X-quang", "action": "LLaVA", "action_input": {"question": "Describe abnormalities in this chest X-ray"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nTìm vùng tim to (cardiomegaly)."},
                {"role": "assistant", "content": '{"thought": "Detect cardiomegaly", "action": "GroundingDINO", "action_input": {"prompt": "enlarged heart cardiomegaly"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nCó dấu hiệu viêm phổi không?"},
                {"role": "assistant", "content": '{"thought": "Kiểm tra pneumonia", "action": "GroundingDINO", "action_input": {"prompt": "pneumonia consolidation"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nSegment vùng bất thường đã detect."},
                {"role": "assistant", "content": '{"thought": "Segment detected regions", "action": "MedSAM", "action_input": {"boxes": "from_previous_detection"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "Cardiomegaly là gì?"},
                {"role": "assistant", "content": '{"thought": "Câu hỏi lý thuyết, dùng RAG", "action": "RAG", "action_input": {"query": "cardiomegaly definition causes treatment"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nWhat abnormalities can you see?"},
                {"role": "assistant", "content": '{"thought": "General abnormality question", "action": "LLaVA", "action_input": {"question": "What abnormalities can you see?"}}'}
            ]
        },
        {
            "image": None,
            "conversations": [
                {"role": "user", "content": "<image>\nIs there any pleural effusion?"},
                {"role": "assistant", "content": '{"thought": "Check for effusion", "action": "GroundingDINO", "action_input": {"prompt": "pleural effusion"}}'}
            ]
        }
    ]
    
    # Expand sample data
    expanded_samples = SAMPLE_DATA * 50  # 400 samples
    all_examples.extend(expanded_samples)
    print(f"✅ Added sample data: {len(expanded_samples)} examples")

# Shuffle
random.seed(42)
random.shuffle(all_examples)

# Split train/val
split_idx = int(len(all_examples) * 0.9)
train_data = all_examples[:split_idx]
val_data = all_examples[split_idx:]

# Save
train_file = SFT_DIR / "train.jsonl"
val_file = SFT_DIR / "val.jsonl"

with open(train_file, 'w', encoding='utf-8') as f:
    for item in train_data:
        # Clean item (remove extra fields)
        clean_item = {
            "image": item.get("image"),
            "conversations": item["conversations"]
        }
        f.write(json.dumps(clean_item, ensure_ascii=False) + '\n')

with open(val_file, 'w', encoding='utf-8') as f:
    for item in val_data:
        clean_item = {
            "image": item.get("image"),
            "conversations": item["conversations"]
        }
        f.write(json.dumps(clean_item, ensure_ascii=False) + '\n')

print(f"\n" + "="*60)
print("📊 SFT DATASET SUMMARY")
print("="*60)
print(f"📁 Output directory: {SFT_DIR}")
print(f"📄 Train samples: {len(train_data)}")
print(f"📄 Val samples: {len(val_data)}")
print(f"📄 Total: {len(all_examples)}")

# Action distribution
from collections import Counter
actions = []
for ex in all_examples:
    try:
        content = ex['conversations'][1]['content']
        plan = json.loads(content)
        actions.append(plan.get('action', 'unknown'))
    except:
        actions.append('parse_error')

action_dist = Counter(actions)
print(f"\n📊 Action Distribution:")
for action, count in action_dist.most_common():
    pct = count / len(all_examples) * 100
    print(f"   - {action}: {count} ({pct:.1f}%)")

---
## 3️⃣ 🚀 Training Configuration

In [ ]:
# @title 3.1 ⚙️ Training Config
from dataclasses import dataclass
from typing import List

@dataclass
class SFTTrainingConfig:
    # Model - LLaVA-Med for medical imaging
    base_model: str = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
    
    # LoRA
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    target_modules: List[str] = None
    
    # Training
    num_epochs: int = 3
    batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    max_seq_length: int = 2048
    
    # Quantization
    load_in_4bit: bool = True
    bf16: bool = True
    
    # Output
    output_dir: str = "checkpoints/sft_adapter"
    hub_model_id: str = "ngnam1104/trimedagent-sft-v1"
    
    def __post_init__(self):
        if self.target_modules is None:
            self.target_modules = ["q_proj", "v_proj", "k_proj", "o_proj", 
                                   "gate_proj", "up_proj", "down_proj"]

config = SFTTrainingConfig()
print("✅ Config ready!")
print(f"   Base: {config.base_model}")
print(f"   LoRA: r={config.lora_r}, α={config.lora_alpha}")
print(f"   Epochs: {config.num_epochs}")

In [ ]:
# @title 3.2 📚 Load Model with LoRA (Standard - No Unsloth)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import torch

print(f"🚀 Loading {config.base_model}...")
print("   (LLaVA-Med - specialized for medical imaging)")

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.load_in_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if config.bf16 else torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Tokenizer loaded!")

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    config.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if config.bf16 else torch.float16
)

print("✅ Base model loaded!")

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# LoRA config
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable params
trainable, total = model.get_nb_trainable_parameters()
print(f"\n✅ LoRA applied!")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# @title 3.3 📊 Prepare Dataset
from torch.utils.data import Dataset
import json

SYSTEM_PROMPT = """You are TriMed-Agent, a medical AI assistant.
Analyze medical images and respond with structured JSON plans.
Format:
{
    "thought": "your reasoning",
    "action": "tool name (GroundingDINO, MedSAM, RAG, or answer)",
    "action_input": {"param": "value"}
}"""

class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        
        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    self.data.append(json.loads(line))
                    
        print(f"Loaded {len(self.data)} examples")
    
    def __len__(self):
        return len(self.data)
    
    def _format_conversation(self, item):
        convs = item.get('conversations', [])
        parts = [f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"]
        
        for turn in convs:
            if turn['role'] == 'user':
                parts.append(f"{turn['content']} [/INST] ")
            else:
                parts.append(f"{turn['content']} </s>")
        
        return "".join(parts)
    
    def __getitem__(self, idx):
        text = self._format_conversation(self.data[idx])
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze().clone()
        }

train_dataset = SFTDataset(
    "data/sft_dataset/train.jsonl",
    tokenizer,
    config.max_seq_length
)

print(f"✅ Dataset ready: {len(train_dataset)} samples")

---
## 4️⃣ 🔥 Start Training

In [ ]:
# @title 4.1 🚂 Run Training with Loss Logging
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
import matplotlib.pyplot as plt

# Custom callback để log losses
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.steps = []
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])
            self.steps.append(state.global_step)

loss_logger = LossLoggerCallback()

# Training arguments
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=0.01,
    bf16=config.bf16,
    logging_steps=5,  # Log mỗi 5 steps để có nhiều điểm
    save_steps=50,
    save_total_limit=3,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    logging_first_step=True,
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

# Trainer với callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[loss_logger]
)

print("🚂 Starting SFT Training...")
print(f"   Model: {config.base_model}")
print(f"   Samples: {len(train_dataset)}")
print(f"   Epochs: {config.num_epochs}")
print(f"   Batch: {config.batch_size} x {config.gradient_accumulation_steps}")
print(f"   LR: {config.learning_rate}")
print("="*60)

# Train
train_result = trainer.train()

print("\n✅ Training Complete!")
print(f"   Final Loss: {train_result.training_loss:.4f}")
print(f"   Total Steps: {train_result.global_step}")

In [ ]:
# @title 4.2 📊 Plot Training Curves (for Report)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss
ax1 = axes[0]
if loss_logger.train_losses:
    ax1.plot(loss_logger.steps, loss_logger.train_losses, 'b-', linewidth=2, label='Training Loss')
    
    # Add smoothed curve
    if len(loss_logger.train_losses) > 5:
        window = min(5, len(loss_logger.train_losses) // 3)
        smoothed = np.convolve(loss_logger.train_losses, np.ones(window)/window, mode='valid')
        smooth_steps = loss_logger.steps[window-1:]
        ax1.plot(smooth_steps, smoothed, 'r-', linewidth=2, alpha=0.7, label='Smoothed')
        
ax1.set_xlabel('Training Steps', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('📉 SFT Training Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Learning Rate Schedule (approximation)
ax2 = axes[1]
total_steps = train_result.global_step
warmup_steps = int(total_steps * config.warmup_ratio)
steps = np.arange(total_steps)

# Cosine with warmup
lr_schedule = []
for step in steps:
    if step < warmup_steps:
        lr = config.learning_rate * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        lr = config.learning_rate * 0.5 * (1 + np.cos(np.pi * progress))
    lr_schedule.append(lr)

ax2.plot(steps, lr_schedule, 'g-', linewidth=2)
ax2.axvline(x=warmup_steps, color='r', linestyle='--', label=f'Warmup End ({warmup_steps} steps)')
ax2.set_xlabel('Training Steps', fontsize=12)
ax2.set_ylabel('Learning Rate', fontsize=12)
ax2.set_title('📈 Learning Rate Schedule (Cosine)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sft_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training curves saved to: sft_training_curves.png")

In [ ]:
# @title 4.3 💾 Save Adapter Locally
from pathlib import Path
import json

output_path = Path(config.output_dir) / "final"
output_path.mkdir(parents=True, exist_ok=True)

# Save model
trainer.save_model(str(output_path))
tokenizer.save_pretrained(str(output_path))

# Save training config & metrics
training_info = {
    'base_model': config.base_model,
    'lora_r': config.lora_r,
    'lora_alpha': config.lora_alpha,
    'target_modules': config.target_modules,
    'num_epochs': config.num_epochs,
    'learning_rate': config.learning_rate,
    'final_loss': train_result.training_loss,
    'total_steps': train_result.global_step,
    'training_samples': len(train_dataset),
}
with open(output_path / "training_config.json", 'w') as f:
    json.dump(training_info, f, indent=2)

# Save loss history for later analysis
loss_history = {
    'steps': loss_logger.steps,
    'losses': loss_logger.train_losses
}
with open(output_path / "loss_history.json", 'w') as f:
    json.dump(loss_history, f, indent=2)

print(f"✅ Adapter saved to: {output_path}")
print(f"   - adapter_model.safetensors")
print(f"   - training_config.json")
print(f"   - loss_history.json")

---
## 5️⃣ 🚀 Push to HuggingFace

In [ ]:
# @title 5.1 📤 Push Adapter to HuggingFace Hub
from huggingface_hub import HfApi

REPO_ID = config.hub_model_id  # e.g., "ngnam1104/trimedagent-sft-v1"

print(f"📤 Pushing to HuggingFace: {REPO_ID}")

try:
    # Push model
    model.push_to_hub(REPO_ID, use_auth_token=True)
    tokenizer.push_to_hub(REPO_ID, use_auth_token=True)
    
    print(f"\n✅ Successfully pushed to: https://huggingface.co/{REPO_ID}")
    
except Exception as e:
    print(f"❌ Push failed: {e}")
    print("\nTry manual upload:")
    print(f"   huggingface-cli upload {REPO_ID} {output_path}")

In [ ]:
# @title 5.2 📋 Create Model Card
model_card = f"""---
license: apache-2.0
tags:
  - medical
  - vision
  - llava
  - lora
  - trimedagent
base_model: {config.base_model}
datasets:
  - custom
language:
  - en
  - vi
---

# TriMedAgent SFT Adapter

LoRA adapter for TriMedAgent - Medical Visual Agent.

## Training Details

- **Base Model**: `{config.base_model}`
- **LoRA Rank**: {config.lora_r}
- **LoRA Alpha**: {config.lora_alpha}
- **Target Modules**: {config.target_modules}
- **Epochs**: {config.num_epochs}

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("{config.base_model}")
model = PeftModel.from_pretrained(base_model, "{REPO_ID}")
```

## Citation

```
@misc{{trimedagent,
  title={{TriMedAgent: Medical Visual Agent}},
  author={{Team}},
  year={{2025}}
}}
```
"""

# Save model card
with open(output_path / "README.md", 'w') as f:
    f.write(model_card)

print("✅ Model card created!")
print(f"\n📋 Next step: Run GRPO/RL training with this SFT adapter")
print(f"   Adapter path: {REPO_ID}")

---
## 🎉 Done!

SFT Training hoàn tất. Adapter đã được push lên HuggingFace.

**Next Steps:**
1. Chạy notebook `02_rl_grpo_training.ipynb` để fine-tune với GRPO
2. Hoặc test adapter ngay với notebook `03_demo.ipynb`